# Test Synthethic Data Generation (SDG) AUG-PE Algorithm

The idea of this algorithm is to generate synthetic copies of sensitive text data using an LLM with an API without the LLM ever reading any of the real data. The algorithm also includes a Differential Privacy (DP) garuantee that ensure that the removal of any one record does not impact the distribution of the data, strenghthening the privacy protections of the algorithm. 

I think in a proper implementation I would set up two classes, one that included the public data and api access and one that had the private data in a hidden variable just to avoid an overlap.

## Step 0 - Input Variables

Generate some fake restuarant reviews for testing and some of the global AUG-PE parameters.

In [1]:
import anthropic
import os
import numpy as np
import pandas as pd
from great_tables import GT, style, loc
import pandas as pd
from sentence_transformers import SentenceTransformer
from huggingface_hub import snapshot_download

In [2]:
# Restaurant reviews
private_data = [
    "The restaurant was fantastic and the service was quick.",
    "Food arrived cold and delivery was very slow.",
    "The pizza crust was so good! But the toppings were a bit disappointing.", 
    "Absolutely loved the ambiance, cozy and warm with excellent staff.",
    "Waited over an hour for my order and it was completely wrong.",
    "The pasta was perfectly cooked but the portion sizes were tiny.",
    "Best burger I have ever had, will definitely be coming back!",
    "The sushi was fresh but the rice was overcooked and mushy.",
    "Rude staff and overpriced food, not worth the trip at all.",
    "Desserts were incredible, the chocolate lava cake was divine."
]

# Global Varibales

N_SYNTHETIC = 10

EPSILON = 4.0        # less noise, paper shows this still gives good privacy
N_CANDIDATES = 100    # more diverse initial pool, note technically this should be n_syn * number of variations.
T = 5                # more iterations to spread out
N_VARIATIONS = 6     # variations per resampled text per iteration


In [3]:
# Connect to anthropic API and load the encoding model
client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

# TODO This model should be sentiment aware - T
#encoder = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
encoder = SentenceTransformer('BAAI/bge-large-en-v1.5')

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 5339.58it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Step 1 - DP_NN_HISTOGRAM

DP_NN_HISTOGRAM constructs a histogram of the nearest neighbours of the real data compared to the synthethic data. Distance is based on the embedding model $\Phi$. Then random gaussian noise $\frac{1}{\epsilon}$ is added to enforce DP privacy. This histogram is then used to draw samples from.

At the most basic level, it is just counting for each synthethic record how many real records is the synthetic record the most similar to. Then we draw from this distribution to get a distribution that should match the distribution of the real data.

In [ ]:
# --- DP-NN HISTOGRAM ---
def dp_nn_histogram(candidates, private_texts, epsilon, n_resample):
    priv_embs = encoder.encode(private_texts, normalize_embeddings=True)
    cand_embs = encoder.encode(candidates, normalize_embeddings=True)

    counts = np.zeros(len(candidates))
    for emb in priv_embs:
        best_idx = int(np.argmax(cand_embs @ emb))
        counts[best_idx] += 1

    print(f"  Raw counts:   {counts.astype(int).tolist()}")
    noisy_counts = counts + np.random.normal(0, 1.0 / epsilon, size=counts.shape)
    noisy_counts = np.clip(noisy_counts, 0, None)
    probs = noisy_counts / noisy_counts.sum()
    print(f"  Resample probs: {[f'{p:.2f}' for p in probs]}")

    top_indices = np.argsort(probs)[::-1][:N_SYNTHETIC]
    resampled = [candidates[i] for i in top_indices]
    print(f"  Resampled pool ({len(resampled)}): {resampled}")
    return resampled

# Step 2 - RANDOM_API

The random api uses a prompt to draw initial candidates from the LLM API. In future, some additional subcategories should be enforced for better initital draws. 

In [5]:
# --- RANDOM API ---
def random_api(n):
    print(f"\n[RANDOM API] Generating {n} initial candidates...")
    prompt = (
        f"Generate {n} restaurant reviews: roughly 1/3 positive, 1/3 negative, 1/3 mixed sentiment. "
        "Each review should be 1-2 sentences. Around 10-15 words long"
        "Make each generated review unique and diverse"
        "Return ONLY a numbered list, one review per line."
    )
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=512,
        messages=[{"role": "user", "content": prompt}]
    )
    lines = [l.strip() for l in response.content[0].text.strip().split("\n") if l.strip()]
    candidates = []
    for line in lines:
        parts = line.split('. ', 1)
        candidates.append(parts[1].strip() if len(parts) == 2 and parts[0].isdigit() else line)
    candidates = candidates[:n]
    for i, c in enumerate(candidates, 1):
        print(f"  {i}. {c}")
    return candidates

## Step 3 - VARIATION API

The VARIATION_API is responsible for drawing a sample from the DP_NN_HISTOGRAM, and then using the LLM API to add some variation to the drawn set of samples. This just helps to add some additional variation to the samples.



In [6]:
# --- VARIATION API ---
# VARIATION_API now takes the full resampled pool
def variation_api(text, n):
    prompt = (
        f"Generate {n} diverse variations of the following restaurant review, "
        "Each review should be 1-2 sentences. Around 10-15 words long"
        "each with the same general sentiment but different wording and style. "
        "Return ONLY a numbered list, one review per line.\n\n"
        f"Review: {text}"
    )
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=512,
        temperature=0.9,# Add some temperature to get more diverse variation
        messages=[{"role": "user", "content": prompt}]
    )
    lines = [l.strip() for l in response.content[0].text.strip().split("\n") if l.strip()]
    candidates = []
    for line in lines:
        parts = line.split('. ', 1)
        candidates.append(parts[1].strip() if len(parts) == 2 and parts[0].isdigit() else line)
    return candidates[:n]

## Step 4 - Run the Loop

Generates random reviews, finds the most similar records, adds variation to them, normally repeats for a certain number of runs. 

In [7]:
# Draw initial candidates
candidates = random_api(N_CANDIDATES)

for t in range(T):
    print(f"\n{'='*60}\nITERATION T={t+1}\n{'='*60}")
    resampled = dp_nn_histogram(candidates, private_data, EPSILON, n_resample=len(private_data))
    
    print(f"\n[VARIATION API]")
    candidates = []
    for text in resampled:
        varied = variation_api([text], n=N_VARIATIONS)
        candidates.extend(varied)
        print(f"  - {varied[0]}")
    
    candidates = resampled + all_variations  # winners carried forward + their variations

# Final synthetic dataset = last iteration's candidates
synthetic_data = candidates


[RANDOM API] Generating 100 initial candidates...
  1. Amazing pasta and exceptional service made our anniversary dinner absolutely perfect tonight.
  2. Food was cold, service terrible, and we waited two hours for mediocre meals.
  3. Great appetizers but main courses disappointing, mixed experience overall for our group.
  4. Outstanding steaks, wonderful atmosphere, definitely coming back next weekend with friends again.
  5. Worst dining experience ever, dirty plates and rude staff ruined our evening completely.
  6. Delicious desserts saved the meal despite bland entrees and slow kitchen service.
  7. Five stars for incredible sushi and attentive waitstaff, exceeded all our expectations.
  8. Overpriced food, tiny portions, and noisy environment made this visit very disappointing.
  9. Good wine selection but average food quality, probably won't return anytime soon.
  10. Phenomenal brunch menu, fresh ingredients, highly recommend the eggs benedict dish especially.
  11. Terrible

NameError: name 'n_synthetic' is not defined

## Step 5 - Display results

Note the algorithm produces a distribution of synthetic data, but each one isn't matched to it's closest real 

In [ ]:
# --- RESULTS TABLE ---
def render_table(private_data, synthetic_data):
    real_embs = encoder.encode(private_data, normalize_embeddings=True)
    syn_embs  = encoder.encode(synthetic_data, normalize_embeddings=True)
    sim_matrix = real_embs @ syn_embs.T

    used = set()
    rows = []
    for i in range(len(private_data)):
        sims = sim_matrix[i].copy()
        for j in used:
            sims[j] = -999
        best_j = int(np.argmax(sims))
        used.add(best_j)
        best_sim = sim_matrix[i, best_j]
        conf = "High" if best_sim >= 0.8 else "Medium" if best_sim >= 0.6 else "Low"
        rows.append({
            "Real Review":        private_data[i],
            "Synthetic Match":    synthetic_data[best_j],
            "Cosine Sim":         round(float(best_sim), 3),
            "Confidence":         conf
        })

    df = pd.DataFrame(rows)

    tbl = (
        GT(df)
        .tab_header(
            title="AUG-PE Results",
            subtitle=f"DP ε={EPSILON} — Budget spent: {EPSILON * T:.2f} ({T} iterations)"
        )
        .tab_style(
            style=style.fill(color="#d4edda"),
            locations=loc.body(rows=df.index[df["Confidence"] == "High"].tolist())
        )
        .tab_style(
            style=style.fill(color="#fff3cd"),
            locations=loc.body(rows=df.index[df["Confidence"] == "Medium"].tolist())
        )
        .tab_style(
            style=style.fill(color="#f8d7da"),
            locations=loc.body(rows=df.index[df["Confidence"] == "Low"].tolist())
        )
        .fmt_number(columns="Cosine Sim", decimals=3)
        .cols_width({"Real Review": "35%", "Synthetic Match": "35%", "Cosine Sim": "15%", "Confidence": "15%"})
    )
    return tbl

# At the end of the script:
render_table(private_data, candidates)

GT(_tbl_data=                                         Real Review  \
0  The restaurant was fantastic and the service w...   
1      Food arrived cold and delivery was very slow.   
2  The pizza crust was so good! But the toppings ...   
3  Absolutely loved the ambiance, cozy and warm w...   
4  Waited over an hour for my order and it was co...   
5  The pasta was perfectly cooked but the portion...   
6  Best burger I have ever had, will definitely b...   
7  The sushi was fresh but the rice was overcooke...   
8  Rude staff and overpriced food, not worth the ...   
9  Desserts were incredible, the chocolate lava c...   

                                     Synthetic Match  Cosine Sim Confidence  
0  Wonderful restaurant offering effortless servi...       0.783     Medium  
1  Food arrived ice cold after an extremely long ...       0.852       High  
2  Five stars for this place! Great waiters and s...       0.561        Low  
3  Exceptional dining experience - the staff impr...       0.680     Medium  
4  Terrible experience: frigid food took ages to ...       0.747     Medium  
5  Incredible service paired with salmon that was...       0.565        Low  
6  Fantastic restaurant experience with attentive...       0.672     Medium  
7  Awful service combined with freezing food that...       0.564        Low  
8  Terrible restaurant visit featuring frozen mea...       0.774     Medium  
9  Phenomenal dining with impeccable service and ...       0.673     Medium  , _body=<great_tables._gt_data.Body object at 0x78a2989d1f90>, _boxhead=Boxhead([ColInfo(var='Real Review', type=<ColInfoTypeEnum.default: 1>, column_label='Real Review', column_align='left', column_width='35%'), ColInfo(var='Synthetic Match', type=<ColInfoTypeEnum.default: 1>, column_label='Synthetic Match', column_align='left', column_width='35%'), ColInfo(var='Cosine Sim', type=<ColInfoTypeEnum.default: 1>, column_label='Cosine Sim', column_align='right', column_width='15%'), ColInfo(var='Confidence', type=<ColInfoTypeEnum.default: 1>, column_label='Confidence', column_align='left', column_width='15%')]), _stub=<great_tables._gt_data.Stub object at 0x78a298a0f910>, _spanners=Spanners([]), _heading=Heading(title='AUG-PE Results', subtitle='DP ε=4.0 — Budget spent: 20.00 (5 iterations)', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x78a2989d3b90>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x78a2989d28d0>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='Real Review', rownum=1, colnum=None, styles=[CellStyleFill(color='#d4edda')]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='Synthetic Match', rownum=1, colnum=None, styles=[CellStyleFill(color='#d4edda')]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='Cosine Sim', rownum=1, colnum=None, styles=[CellStyleFill(color='#d4edda')]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='Confidence', rownum=1, colnum=None, styles=[CellStyleFill(color='#d4edda')]), StyleInfo(locname=LocBody(columns=None, rows=[0, 3, 4, 6, 8, 9], mask=None), grpname=None, colname='Real Review', rownum=0, colnum=None, styles=[CellStyleFill(color='#fff3cd')]), StyleInfo(locname=LocBody(columns=None, rows=[0, 3, 4, 6, 8, 9], mask=None), grpname=None, colname='Real Review', rownum=3, colnum=None, styles=[CellStyleFill(color='#fff3cd')]), StyleInfo(locname=LocBody(columns=None, rows=[0, 3, 4, 6, 8, 9], mask=None), grpname=None, colname='Real Review', rownum=4, colnum=None, styles=[CellStyleFill(color='#fff3cd')]), StyleInfo(locname=LocBody(columns=None, rows=[0, 3, 4, 6, 8, 9], mask=None), grpname=None, colname='Real Review', rownum=6, colnum=None, styles=[CellStyleFill(color='#fff3cd')]), StyleInfo(locname=LocBody(columns=None, rows=[0, 3, 4, 6, 8, 9], mask=None), grpname=None, colname='Real R

## Improvements

- This was put together really quickly, need to refer to the [paper](https://alphapav.github.io/augpe-dpapitext/) or better use their code. There are defintely issues.
- Draw multiples Synethtic samples per real dataset to get a better distribution
- Add some tests on a benchmark like a sentiment analysis.
- I added a sentiment penalty for comparison which is a little hacky. Instead there must be some sentiment aware encodings?
- Do some fine tuning on the encoding model?
- Instead of voting for the nearest neighbour, spread the vote weighted across all neighbours? Should still be the same DP?